# 02 · Spatial autocorrelation and blockCV folds: review before modelling

**Brief sections 7–9.** Run after `R/01_spatial_autocorrelation.R` and `R/02_blockcv_folds.R`. The R outputs are imported here, never regenerated. This notebook:

1. shows the spatial-autocorrelation results: variograms, practical ranges, Moran's I;
2. validates the outer fold files for 100, 200 and 300 m blocks;
3. checks the actual test-to-nearest-training distances, which a block size does not guarantee;
4. checks whether re-using the four remaining outer folds as inner folds gives adequate spatial and class representation;
5. quantifies what a spatial purge would remove;
6. records the primary-block-size decision **before** any model is fitted.

In [ ]:
import os, sys, json
from pathlib import Path

# Run mode: "full" for the experiment, "smoke" for the reduced software validation run.
# os.environ["SIEMREAP_MODE"] = "smoke"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "project.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

from siemreap import config
cfg = config.load_config(ROOT)
MODE = config.run_mode()
print(config.smoke_banner())

from siemreap import data, folds as F, geo
AGG = config.stage_dir(cfg, "04_review", "aggregate")
mt = pd.read_csv(config.processed_dir(cfg) / "cells_all.csv")
sac = config.stage_dir(cfg, "04_spatial_autocorrelation", "aggregate")
bcv = config.stage_dir(cfg, "04_blockcv_folds", "aggregate")

## 1. Spatial autocorrelation (from R)

* **Range parameter vs practical range.** The *range parameter* is a model coefficient. The *practical range* is the distance at which the fitted variogram reaches 95 % of its sill: about 3 × the parameter for an exponential model and √3 × for a Gaussian one. The practical range is the effective correlation distance.
* **Unresolved slices.** `resolved = False` marks slices whose fitted range is shorter than the first lag class (25 m). For those, the variogram is flat at every distance the boreholes sample.
* **Exploratory only.** These are response variograms, which serve as exploratory proxies for residual autocorrelation.

In [ ]:
vp = pd.read_csv(sac / "variogram_parameters.csv"); display(vp)
display(pd.read_csv(sac / "morans_I_by_slice.csv"))
display(pd.read_csv(sac / "blockcv_spatial_autocor_by_slice.csv"))
summary_sac = json.loads((sac / "spatial_autocorrelation_summary.json").read_text())
print(json.dumps(summary_sac, indent=1))
for f in ["fig_variograms_by_slice.png", "fig_pooled_standardised_variogram.png", "fig_vertical_variogram.png"]:
    display(Image(filename=str(sac / f), width=720))

## 2. Outer folds: integrity and composition

In [ ]:
blocks = cfg["blockcv"]["block_sizes_m"]
outer = {b: F.load_outer(cfg, b) for b in blocks}
checks = {b: F.validate_outer(outer[b], mt, cfg) for b in blocks}       # raises on any violation
for b in blocks:
    print(f"{b} m:", {k: v for k, v in checks[b].items() if k != "per_fold"})
    display(pd.DataFrame(checks[b]["per_fold"]).assign(block_m=b))

## 3. Test-to-nearest-training distances (Python check of the R table)

In [ ]:
rows = []
for b in blocks:
    for k in range(1, cfg["blockcv"]["k"] + 1):
        for sc in ("A", "B"):
            tr, te = F.split(mt, outer[b], k, sc)
            d = F.test_train_distances(tr, te)
            rows.append({"block_m": b, "fold": k, "scenario": sc, "test_boreholes": len(d), "min_m": d.min(),
                         "median_m": d.median(), "max_m": d.max(), "share_below_50m": (d < 50).mean(),
                         "share_below_block": (d < b).mean()})
dist_py = pd.DataFrame(rows).round(2)
dist_r = pd.read_csv(bcv / "test_to_train_distance_by_fold.csv")
cmp = dist_py.merge(dist_r, left_on=["block_m", "fold", "scenario"], right_on=["block_size_m", "fold", "scenario"],
                    suffixes=("_py", "_r"))
assert np.allclose(cmp.median_m_py, cmp.median_m_r, atol=0.01), "Python and R distances disagree"
display(dist_py.groupby(["block_m", "scenario"])[["min_m", "median_m", "max_m", "share_below_50m", "share_below_block"]].mean().round(2))
dist_py.to_csv(AGG / "test_to_train_distances.csv", index=False)

A block size does **not** guarantee that size as the minimum separation: a test borehole near a block edge can have a training borehole just across the edge. The `share_below_block` column measures this directly.

## 4. Inner folds: adequacy of re-using the remaining outer folds

In [ ]:
rows = []
for b in blocks:
    for k in range(1, cfg["blockcv"]["k"] + 1):
        for sc in ("A", "B"):
            tr, _ = F.split(mt, outer[b], k, sc)
            g = tr.BH_Code.map(outer[b].fold).astype(int).values.copy(); g[g == 0] = -1
            a = F.adequacy(tr, g, cfg)
            rows.append({"block_m": b, "outer_fold": k, "scenario": sc, "all_ok": bool(a.ok.all()),
                         "min_boreholes": a.boreholes.min(), "min_cells": a.cells.min(),
                         "clay_share_range": f"{a.clay_share.min():.2f}-{a.clay_share.max():.2f}"})
adq = pd.DataFrame(rows); display(adq)
adq.to_csv(AGG / "inner_fold_adequacy.csv", index=False)
print("Outer folds needing the R fallback inner folds:", int((~adq.all_ok).sum()))

## 5. Spatial purge (separate analysis): training boreholes removed

In [ ]:
rows = []
pb = cfg["blockcv"]["primary_block_m"]
for d in cfg["experiment"]["purge_distances_m"]:
    for k in range(1, cfg["blockcv"]["k"] + 1):
        for sc in ("A", "B"):
            tr0, _ = F.split(mt, outer[pb], k, sc)
            tr1, _ = F.split(mt, outer[pb], k, sc, purge_m=d)
            rows.append({"purge_m": d, "fold": k, "scenario": sc, "train_boreholes": tr0.BH_Code.nunique(),
                         "removed": tr0.BH_Code.nunique() - tr1.BH_Code.nunique()})
purge = pd.DataFrame(rows); display(purge.groupby(["purge_m", "scenario"])[["train_boreholes", "removed"]].mean().round(1))
purge.to_csv(AGG / "purge_effect.csv", index=False)

## 6. Primary block size: decision recorded before modelling

The brief plans **100 m** as primary, subject to methodological review, and forbids choosing it by model score. The rule in the design report (section 4.3) uses only the variogram range *r̃* and the Wasserstein distance *W* between each design's test-to-train distances and the prediction-domain distances (`blockCV::cv_distance`). `R/02_blockcv_folds.R` evaluates the rule and records it in `block_size_rule.json`.

The configured `primary_block_m` is the one used. If the rule disagrees, change `config/project.toml` **now**, before notebook 03. All three block sizes are run either way.

In [ ]:
W = pd.read_csv(bcv / "wasserstein_vs_prediction_domain.csv"); display(W)
rule = json.loads((bcv / "block_size_rule.json").read_text()); print(json.dumps(rule, indent=1))
for b in blocks:
    display(Image(filename=str(bcv / f"fig_cv_distance_{b}m.png"), width=560))
primary = cfg["blockcv"]["primary_block_m"]
decision = {"configured_primary_m": primary, "rule_choice_m": rule.get("rule_choice_m"),
            "agrees": primary == rule.get("rule_choice_m"), "r_tilde_m": rule.get("r_tilde_m"),
            "W_cv": rule.get("W_cv"), "note": "recorded before any model is fitted (notebook 02)"}
if not decision["agrees"]:
    print(f"NOTE: the configured primary ({primary} m) differs from the rule's choice ({rule.get('rule_choice_m')} m). "
          "Keep it only with a documented reason, or change config/project.toml before notebook 03.")
(AGG / "primary_block_decision.json").write_text(json.dumps(decision, indent=1, default=str))